# fhkit.at — Chakra v3, five locales, emailwerk

The acceptance gate for the migration. It reads the **built** `public/`
directory rather than the source, because every failure this migration can
produce is silent at build time:

- an unresolvable Chakra v3 token is not an error. It is emitted as the
  literal string inside a well-formed declaration, and the browser drops the
  rule without a word. `yarn build` stays green.
- a length without a unit (`font-size:30`) is invalid CSS and is dropped the
  same way. v2 completed such values, v3 does not.
- a page that never fanned out into its locales still builds; it is simply
  missing, and nothing counts the pages for you.
- jaen seeds `jaenPageMetadata.title` from `pageConfig.label`, so an
  unlocalized title is a real English string, not an empty one.

Run `yarn build` first. The notebook asserts, it does not build.


In [ ]:
import json, os, re, subprocess, sys
from pathlib import Path

SITE = Path(os.environ.get('FHKIT_SITE_DIR', '..')).resolve()
PUBLIC = SITE / 'public'
LOCALES = os.environ.get('FHKIT_LOCALES', 'de,en,sl,it,ja').split(',')
DEFAULT_LOCALE = os.environ.get('FHKIT_DEFAULT_LOCALE', 'de')

RESULTS = []

def check(name, ok, detail=''):
    RESULTS.append({'name': name, 'pass': bool(ok), 'detail': str(detail)})
    print(('PASS  ' if ok else 'FAIL  ') + name + (f'  — {detail}' if detail else ''))
    return bool(ok)

def html_files():
    # Files only: the 404 page's data lives in a directory named page-data/404.html.
    return sorted(p for p in PUBLIC.rglob('*.html') if p.is_file())

check('public/ exists', PUBLIC.is_dir(), PUBLIC)
check('public/ is not empty', len(html_files()) > 0, f'{len(html_files())} html files')


## Localized page generation

The default locale keeps the bare path; every other locale lives under its
prefix. System routes (`/cms`, `/login`, `/settings`, `/emailwerk`) are never
localized, and a prefixed clone of one is a defect, not a bonus page.


In [ ]:
routes = {p.parent.relative_to(PUBLIC).as_posix() for p in PUBLIC.rglob('index.html')}
routes = {('/' + r if r != '.' else '/') for r in routes}

SITE_ROUTES = ['/', '/imprint']

for route in SITE_ROUTES:
    check(f'{route} exists in the default locale', route.rstrip('/') or '/' in routes or route in routes, route)

for loc in [l for l in LOCALES if l != DEFAULT_LOCALE]:
    for route in SITE_ROUTES:
        target = f'/{loc}' if route == '/' else f'/{loc}{route}'
        check(f'{target} exists', target in routes, target)

SYSTEM = ['cms', 'login', 'logout', 'settings', 'signup', 'emailwerk']
leaked = [r for r in routes
          for loc in LOCALES if loc != DEFAULT_LOCALE
          for s in SYSTEM if r.startswith(f'/{loc}/{s}')]
check('no locale-prefixed system routes', not leaked, leaked[:5])


## html lang, hreflang and og:locale

Emitted by gatsby-plugin-jaen's `onRenderBody` from the page context. All
three need `siteUrl` on the plugin options; without it they are skipped with
a warning that is easy to miss in a long build log.


In [ ]:
def read(path):
    return (PUBLIC / path).read_text(encoding='utf8')

for loc in LOCALES:
    rel = 'index.html' if loc == DEFAULT_LOCALE else f'{loc}/index.html'
    h = read(rel)
    m = re.search(r'<html[^>]*lang="([^"]+)"', h)
    check(f'/{"" if loc == DEFAULT_LOCALE else loc} carries lang={loc}',
          m and m.group(1) == loc, m.group(1) if m else 'no lang attribute')

home = read('index.html')
alts = set(re.findall(r'rel="alternate"\s+hrefLang="([^"]+)"', home))
check('home page links every locale as an alternate',
      set(LOCALES) <= alts, sorted(alts))
check('home page carries x-default', 'x-default' in alts, sorted(alts))
check('home page carries og:locale', 'og:locale' in home)


## Sitemap

Written by gatsby-source-jaen in `onPostBuild`, replacing gatsby-plugin-sitemap,
which had neither locale awareness nor system-route knowledge.


In [ ]:
sm = PUBLIC / 'sitemap.xml'
check('sitemap.xml exists', sm.is_file())

if sm.is_file():
    x = sm.read_text(encoding='utf8')
    hl = set(re.findall(r'hreflang="([^"]+)"', x))
    check('sitemap carries every locale', set(LOCALES) <= hl, sorted(hl))
    check('sitemap carries x-default', 'x-default' in hl)
    check('sitemap lists no system route',
          not re.search(r'<loc>[^<]*/(cms|login|logout|settings|signup|emailwerk)/', x))
    check('sitemap has no placeholder host', 'yourdomain.tld' not in x)


## Tokens that did not resolve

The cheapest net in the whole migration. A Chakra v3 token path that does not
resolve reaches the stylesheet as its own literal text, so grepping the built
HTML for the site's own token namespaces finds every one of them at once.

This is what caught the four dead paths in `Filesystem.tsx`, the `boxShadow`
in `ImageCard.tsx` read out of the colors namespace, and two namespaces that
are commented out of the token tree entirely.


In [ ]:
NAMESPACES = ['components', 'shared', 'leftNav', 'rightNav', 'topNav',
              'footer', 'features', 'pq', 'main', 'pages', 'views', 'modals']
pattern = re.compile(r':(' + '|'.join(NAMESPACES) + r')\.[A-Za-z.]+')

leaks = {}
for f in html_files():
    for m in pattern.findall(f.read_text(encoding='utf8')):
        leaks.setdefault(m, set()).add(f.relative_to(PUBLIC).as_posix())

check('no literal token path in the built HTML', not leaks,
      '; '.join(sorted(leaks)[:5]))


## Lengths without a unit

`font-size:30` is not a length and the declaration is discarded. Unitless
`line-height` is deliberately excluded: there it is a valid multiplier and the
whole theme relies on it.


In [ ]:
bad = {}
for f in html_files():
    for m in re.findall(r'(font-size|height|width|min-height|max-width):[0-9]+[;"]',
                        f.read_text(encoding='utf8')):
        pass
    for m in re.findall(r'font-size:[0-9]+[;"]', f.read_text(encoding='utf8')):
        bad.setdefault(m, set()).add(f.relative_to(PUBLIC).as_posix())

check('no unitless font-size', not bad, '; '.join(sorted(bad)[:5]))


## Page titles

gatsby-source-jaen seeds every JaenPage with
`jaenPageMetadata.title = pageConfig.label`, and jaen resolves
`metadata.title || label || siteMetadata.title`, so the seeded placeholder
always outranks a localized default unless the site's own Head recognises it.
Before that was fixed this build shipped `root`, `Home Page` and
`Imprint page` as titles.


In [ ]:
PLACEHOLDERS = {'root', 'Home Page', 'Imprint page', 'Documentation page',
                'Experiments page', 'index', 'Index'}

titles = {}
for f in PUBLIC.rglob('index.html'):
    rel = '/' + f.parent.relative_to(PUBLIC).as_posix().replace('.', '')
    m = re.search(r'<title[^>]*>([^<]*)', f.read_text(encoding='utf8'))
    if m:
        titles[rel] = m.group(1).strip()

site_titles = {r: t for r, t in titles.items()
               if not any(s in r for s in ['/cms', '/login', '/logout', '/settings',
                                           '/signup', '/emailwerk'])}

seeded = {r: t for r, t in site_titles.items() if t in PLACEHOLDERS}
check('no page ships a seeded placeholder title', not seeded, seeded)
check('every site page has a title', all(site_titles.values()),
      [r for r, t in site_titles.items() if not t])

for r, t in sorted(site_titles.items()):
    print(f'  {r:32} {t}')


## Translations actually render

A catalogue that exists but is never read looks identical to one that works,
because react-intl falls back to the `defaultMessage` in the source, which is
German. So the check is positive: each locale must contain its own string.


In [ ]:
MARKERS = {
    'de': 'Jetzt anfragen',
    'en': 'Get in touch',
    'sl': 'Pišite nam',
    'it': 'Richiedi ora',
    'ja': 'お問い合わせはこちら',
}

for loc, marker in MARKERS.items():
    if loc not in LOCALES:
        continue
    rel = 'index.html' if loc == DEFAULT_LOCALE else f'{loc}/index.html'
    h = read(rel)
    check(f'{loc} renders its own navigation', marker in h, marker)

# and the German string must NOT survive into the others
for loc in [l for l in LOCALES if l != DEFAULT_LOCALE]:
    h = read(f'{loc}/index.html')
    check(f'{loc} does not fall back to German nav',
          MARKERS['de'] not in h)


## Contact form wiring

Static checks on the source. Pressing Send is checked in `fhkit-browser.ipynb`,
with emailwerk's answer played inside the browser so that no mail goes out.
See `docs/emailwerk-setup.md` for the server side.

The form posts the mutation itself, with one plain fetch. The
`sendTemplateMail` helper in jaen goes through gqty, whose default retry
policy repeats a send that failed in the network, and a repeat after
emailwerk had accepted the send would deliver the enquiry twice.


In [ ]:
contact = (SITE / 'src/services/contact.tsx').read_text(encoding='utf8')

check("contact posts sendTemplateMail itself, not through jaen's gqty client",
      'sendTemplateMail(args:' in contact
      and re.search(r'''from\s+['"]gatsby-jaen-emailwerk['"]''', contact) is None)
check('contact sends only replyTo on the envelope',
      'replyTo' in contact and re.search(r'envelope(Override)?:\s*\{[^}]*\bto\b', contact) is None)
check('contact makes one request per send and does not send the confirmation itself',
      contact.count('sendTemplateMail(args:') == 1 and contact.count('fetch(') == 1)
check('contact carries the localized mail text',
      't_subject' in contact and 'confirmationText' in contact)
template = re.findall(r"CONTACT_TEMPLATE_ID\s*=\s*'([^']*)'", contact)
check("contact posts to fhkit's own template, \"fhkit Contact\"",
      template == ['cmv2fl2cu14ebps2p05nwpoev'], template)
check('the dialog shows the outcome, no toast and no mailto fallback',
      'ContactSendResult' in contact and 'toast(' not in contact
      and 'window.location.href =' not in contact)

modal = (SITE / 'src/components/ContactModal/CheckboxStyled.tsx').read_text(encoding='utf8')
check('the consent checkbox reports through onCheckedChange',
      'onCheckedChange' in modal,
      'v2 wrote onChange/isChecked, which v3 ignores, so consent never validated')

config = (SITE / 'gatsby-config.ts').read_text(encoding='utf8')
check('emailwerk plugin is registered', 'gatsby-jaen-emailwerk' in config)
# the placeholder must be gone from the VALUE, not from the prose around
# it: the comment that records the fix names the old host on purpose.
site_urls = re.findall(r"siteUrl:\s*[`'\"]([^`'\"]+)", config)
check('siteUrl is set to fhkit.at', site_urls and all('fhkit.at' in u for u in site_urls),
      site_urls)


## Open items

Not failures. These are the values that only exist outside this repository,
and the check exists so they cannot be forgotten quietly.


In [ ]:
todos = []
for rel in ['gatsby-config.ts', 'src/services/contact.tsx']:
    t = (SITE / rel).read_text(encoding='utf8')
    for line in t.splitlines():
        if 'TODO' in line:
            todos.append(f'{rel}: {line.strip()[:90]}')

print(f'{len(todos)} open item(s):')
for t in todos:
    print('  -', t)


In [ ]:
passed = sum(1 for r in RESULTS if r['pass'])
total = len(RESULTS)
print(f'\n{passed}/{total} checks passed')

out = SITE / 'tests' / 'results-fhkit-migration.json'
out.write_text(json.dumps({'passed': passed, 'total': total,
                           'checks': RESULTS}, indent=1, ensure_ascii=False),
               encoding='utf8')
print('written to', out)

assert passed == total, f'{total - passed} check(s) failed'
